In [1]:
import pandas as pd
import seaborn as sns
import numpy as np

pd.set_option("display.max_columns", None)

In [2]:
df = pd.read_csv("https://raw.githubusercontent.com/mwaskom/seaborn-data/refs/heads/master/titanic.csv")

In [3]:
print(df.shape)

(891, 15)


In [4]:
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [5]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
print("Columns:", list(df.columns))

Columns: ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town', 'alive', 'alone']


In [6]:
duplicate_count = df.duplicated().sum()
print("Duplicate rows found:", duplicate_count)


Duplicate rows found: 107


In [7]:
df = df.drop_duplicates()

In [8]:
numeric_cols = ["age", "fare", "sibsp", "parch"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [9]:
invalid_age_count = ((df["age"] < 0) | (df["age"] > 100)).sum()
df.loc[(df["age"] < 0) | (df["age"] > 100), "age"] = np.nan
print("Impossible age values reset to NaN:", invalid_age_count)

Impossible age values reset to NaN: 0


In [10]:
print("Missing values BEFORE imputation:")
print(df.isna().sum())

Missing values BEFORE imputation:
survived         0
pclass           0
sex              0
age            106
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           582
embark_town      2
alive            0
alone            0
dtype: int64


In [11]:
df["age"] = df.groupby(["pclass", "sex"])["age"].transform(
    lambda x: x.fillna(x.median()))

In [12]:
df["age"] = df["age"].fillna(df["age"].median())

In [13]:
print("Missing age values remaining:", df["age"].isna().sum())

Missing age values remaining: 0


In [14]:
print(df.groupby(["pclass", "sex"])["age"].median())

pclass  sex   
1       female    35.0
        male      40.0
2       female    28.0
        male      30.5
3       female    22.0
        male      25.0
Name: age, dtype: float64


In [15]:
embarked_mode = df["embarked"].mode()[0]
df["embarked"] = df["embarked"].fillna(embarked_mode)
print("Filled 'embarked' missing values with mode:", embarked_mode)

Filled 'embarked' missing values with mode: S


In [16]:
embark_town_mode = df["embark_town"].mode()[0]
df["embark_town"] = df["embark_town"].fillna(embark_town_mode)
print("Filled 'embark_town' missing values with mode:", embark_town_mode)

Filled 'embark_town' missing values with mode: Southampton


In [17]:
print("Missing values")
print(df.isna().sum())

print("\nShape:", df.shape)

Missing values
survived         0
pclass           0
sex              0
age              0
sibsp            0
parch            0
fare             0
embarked         0
class            0
who              0
adult_male       0
deck           582
embark_town      0
alive            0
alone            0
dtype: int64

Shape: (784, 15)


In [18]:
df["deck"] = df["deck"].astype(object)
df["deck"] = df["deck"].fillna("Unknown")

In [19]:
print("Deck value counts after fixing:")
print(df["deck"].value_counts())
print("\nMissing deck values remaining:", df["deck"].isna().sum())

Deck value counts after fixing:
deck
Unknown    582
C           59
B           46
D           33
E           32
A           15
F           13
G            4
Name: count, dtype: int64

Missing deck values remaining: 0


In [20]:
invalid_fare_mask = (df["fare"] <= 0) | (df["fare"].isna())
invalid_fare_count = invalid_fare_mask.sum()

df["fare"] = df.groupby("pclass")["fare"].transform(
    lambda x: x.mask(x <= 0, x.median()))

In [21]:
df["fare"] = df["fare"].fillna(df["fare"].median())

In [22]:
print("Non-positive/missing fares fixed:", invalid_fare_count)
print("Missing fare values remaining:", df["fare"].isna().sum())
print("\nMedian fare by class:")
print(df.groupby("pclass")["fare"].median())

Non-positive/missing fares fixed: 10
Missing fare values remaining: 0

Median fare by class:
pclass
1    60.73125
2    18.75000
3     8.05000
Name: fare, dtype: float64


In [23]:
df = df.drop(columns=["alive", "class"])

print("Columns remaining:", list(df.columns))
print("Shape after dropping columns:", df.shape)

Columns remaining: ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'who', 'adult_male', 'deck', 'embark_town', 'alone']
Shape after dropping columns: (784, 13)


In [24]:
df["family_size"] = df["sibsp"] + df["parch"] + 1
df["is_alone"] = (df["family_size"] == 1).astype(int)

In [25]:
df["age_band"] = pd.cut(
    df["age"],
    bins=[0, 12, 18, 35, 60, 100],
    labels=["Child", "Teen", "Young Adult", "Adult", "Senior"])

In [26]:
df["fare_band"] = pd.qcut(df["fare"], q=4, labels=["Low", "Medium", "High", "Very High"])

In [27]:
print(df[["sibsp", "parch", "family_size", "is_alone", "age", "age_band", "fare", "fare_band"]].head(10))

   sibsp  parch  family_size  is_alone   age     age_band     fare  fare_band
0      1      0            2         0  22.0  Young Adult   7.2500        Low
1      1      0            2         0  38.0        Adult  71.2833  Very High
2      0      0            1         1  26.0  Young Adult   7.9250        Low
3      1      0            2         0  35.0  Young Adult  53.1000  Very High
4      0      0            1         1  35.0  Young Adult   8.0500        Low
5      0      0            1         1  25.0  Young Adult   8.4583     Medium
6      0      0            1         1  54.0        Adult  51.8625  Very High
7      3      1            5         0   2.0        Child  21.0750       High
8      0      2            3         0  27.0  Young Adult  11.1333     Medium
9      1      0            2         0  14.0         Teen  30.0708       High


In [28]:
before = len(df)
df = df.drop_duplicates()
after = len(df)
print(f"Removed {before - after} duplicate rows again. New shape: {df.shape}")

Removed 8 duplicate rows again. New shape: (776, 17)


In [29]:
def validate(df):
    required = {"survived", "pclass", "sex", "age", "fare", "family_size"}
    assert required.issubset(df.columns), "Required columns missing"
    assert df.duplicated().sum() == 0, "Duplicates remain"
    assert df["survived"].isin([0, 1]).all(), "Invalid target values"
    assert df["pclass"].isin([1, 2, 3]).all(), "Invalid passenger class"
    assert df["age"].between(0, 100).all(), "Age outside valid range"
    assert (df["fare"] > 0).all(), "Fare must be positive"
    assert df[list(required)].isna().sum().sum() == 0, "Critical nulls remain"
    return True

is_valid = validate(df)
print("Validation passed:", is_valid)

Validation passed: True


In [30]:
df.to_csv("../data/cleaned_titanic.csv", index=False)
print("Cleaned data saved to ../data/cleaned_titanic.csv")
print("Final shape:", df.shape)
print("\nFinal columns:", list(df.columns))

Cleaned data saved to ../data/cleaned_titanic.csv
Final shape: (776, 17)

Final columns: ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'who', 'adult_male', 'deck', 'embark_town', 'alone', 'family_size', 'is_alone', 'age_band', 'fare_band']


In [31]:
overall_survival_rate = round(df["survived"].mean() * 100, 2)
print(f"Overall survival rate: {overall_survival_rate}%")

Overall survival rate: 41.49%


In [32]:
print("\nSurvival counts:")
print(df["survived"].value_counts())


Survival counts:
survived
0    454
1    322
Name: count, dtype: int64


In [33]:
by_sex = (df.groupby("sex", as_index=False)["survived"]
          .mean()
          .assign(survival_rate=lambda x: (x["survived"] * 100).round(2))
          .drop(columns="survived"))

print(by_sex)

      sex  survival_rate
0  female          73.97
1    male          21.90


In [34]:
by_class = (df.groupby("pclass", as_index=False)["survived"]
            .mean()
            .assign(survival_rate=lambda x: (x["survived"] * 100).round(2))
            .drop(columns="survived"))

print(by_class)

   pclass  survival_rate
0       1          63.68
1       2          50.91
2       3          25.81


In [35]:
print("\nSurvival rate by class AND sex:")
print(df.pivot_table(values="survived", index="pclass", columns="sex", aggfunc="mean").round(3) * 100)


Survival rate by class AND sex:
sex     female  male
pclass              
1         96.8  37.8
2         91.8  18.5
3         46.8  16.1


In [36]:
by_age_band = (df.groupby("age_band", observed=True, as_index=False)["survived"]
               .mean()
               .assign(survival_rate=lambda x: (x["survived"] * 100).round(2))
               .drop(columns="survived"))

print(by_age_band)

      age_band  survival_rate
0        Child          57.35
1         Teen          44.78
2  Young Adult          40.59
3        Adult          39.05
4       Senior          22.73


In [37]:
print("Fare statistics by class:")
print(df.groupby("pclass")["fare"].describe().round(2))

Fare statistics by class:
        count   mean    std    min    25%    50%    75%     max
pclass                                                         
1       212.0  86.29  77.89   5.00  34.50  61.28  96.73  512.33
2       165.0  21.95  13.37  10.50  13.00  18.75  26.00   73.50
3       399.0  13.83  10.65   4.01   7.78   8.05  15.90   69.55


In [38]:
print("\nFare statistics by survival status:")
print(df.groupby("survived")["fare"].describe().round(2))


Fare statistics by survival status:
          count   mean    std   min   25%    50%    75%     max
survived                                                       
0         454.0  24.80  33.57  4.01   7.9  13.00  27.75  263.00
1         322.0  50.24  68.06  6.98  13.0  26.25  62.37  512.33


In [39]:
by_family = (df.groupby("family_size", as_index=False)["survived"]
             .agg(passengers="count", survival_rate="mean"))
by_family["survival_rate"] = (by_family["survival_rate"] * 100).round(2)

print(by_family)

   family_size  passengers  survival_rate
0            1         438          34.02
1            2         154          55.19
2            3         101          57.43
3            4          28          71.43
4            5          13          23.08
5            6          22          13.64
6            7          12          33.33
7            8           6           0.00
8           11           2           0.00


In [40]:
print("Alone vs with family:")
by_alone = (df.groupby("is_alone", as_index=False)["survived"]
            .mean()
            .assign(survival_rate=lambda x: (x["survived"] * 100).round(2))
            .drop(columns="survived"))
print(by_alone)

Alone vs with family:
   is_alone  survival_rate
0         0          51.18
1         1          34.02


In [41]:
embark_crosstab = pd.crosstab(df["embark_town"], df["survived"], normalize="index").round(3) * 100
embark_crosstab.columns = ["Died %", "Survived %"]
print(embark_crosstab)

             Died %  Survived %
embark_town                    
Cherbourg      41.2        58.8
Queenstown     67.2        32.8
Southampton    62.3        37.7


In [42]:
numeric_cols = ["survived", "pclass", "age", "sibsp", "parch", "fare", "family_size", "is_alone"]
correlation_matrix = df[numeric_cols].corr().round(3)
print(correlation_matrix)

             survived  pclass    age  sibsp  parch   fare  family_size  \
survived        1.000  -0.336 -0.075 -0.040  0.068  0.240        0.011   
pclass         -0.336   1.000 -0.399  0.090  0.042 -0.560        0.082   
age            -0.075  -0.399  1.000 -0.291 -0.192  0.116       -0.295   
sibsp          -0.040   0.090 -0.291  1.000  0.380  0.129        0.860   
parch           0.068   0.042 -0.192  0.380  1.000  0.187        0.799   
fare            0.240  -0.560  0.116  0.129  0.187  1.000        0.187   
family_size     0.011   0.082 -0.295  0.860  0.799  0.187        1.000   
is_alone       -0.173   0.108  0.204 -0.608 -0.570 -0.237       -0.710   

             is_alone  
survived       -0.173  
pclass          0.108  
age             0.204  
sibsp          -0.608  
parch          -0.570  
fare           -0.237  
family_size    -0.710  
is_alone        1.000  


In [43]:
def build_insights(df):
    overall = round(df["survived"].mean() * 100, 2)

    by_sex = (df.groupby("sex", as_index=False)["survived"]
              .mean().assign(survival_rate=lambda x: (x["survived"] * 100).round(2))
              .drop(columns="survived"))

    by_class = (df.groupby("pclass", as_index=False)["survived"]
                .mean().assign(survival_rate=lambda x: (x["survived"] * 100).round(2))
                .drop(columns="survived"))

    by_age_band = (df.groupby("age_band", observed=True, as_index=False)["survived"]
                   .mean().assign(survival_rate=lambda x: (x["survived"] * 100).round(2))
                   .drop(columns="survived"))

    by_family = (df.groupby("family_size", as_index=False)["survived"]
                 .agg(passengers="count", survival_rate="mean"))
    by_family["survival_rate"] = (by_family["survival_rate"] * 100).round(2)

    return {
        "overall_survival_rate": overall,
        "by_sex": by_sex.to_dict("records"),
        "by_class": by_class.to_dict("records"),
        "by_age_band": by_age_band.to_dict("records"),
        "by_family_size": by_family.to_dict("records")
    }

insights = build_insights(df)
print(insights)

{'overall_survival_rate': np.float64(41.49), 'by_sex': [{'sex': 'female', 'survival_rate': 73.97}, {'sex': 'male', 'survival_rate': 21.9}], 'by_class': [{'pclass': 1, 'survival_rate': 63.68}, {'pclass': 2, 'survival_rate': 50.91}, {'pclass': 3, 'survival_rate': 25.81}], 'by_age_band': [{'age_band': 'Child', 'survival_rate': 57.35}, {'age_band': 'Teen', 'survival_rate': 44.78}, {'age_band': 'Young Adult', 'survival_rate': 40.59}, {'age_band': 'Adult', 'survival_rate': 39.05}, {'age_band': 'Senior', 'survival_rate': 22.73}], 'by_family_size': [{'family_size': 1, 'passengers': 438, 'survival_rate': 34.02}, {'family_size': 2, 'passengers': 154, 'survival_rate': 55.19}, {'family_size': 3, 'passengers': 101, 'survival_rate': 57.43}, {'family_size': 4, 'passengers': 28, 'survival_rate': 71.43}, {'family_size': 5, 'passengers': 13, 'survival_rate': 23.08}, {'family_size': 6, 'passengers': 22, 'survival_rate': 13.64}, {'family_size': 7, 'passengers': 12, 'survival_rate': 33.33}, {'family_size':